# MTU Notebook Workflow

Upload geospatial data to Mapbox as tilesets. Default mode is **dry run** (no actual upload).

## Quick Start

1. Run **Cell 1** to install `mtu` (skip if already installed).
2. Run **Cell 2** to verify the package is available.
3. Edit **Cell 3** — set your file path, credentials, and tileset info.
4. Run **Cell 4**, **Cell 5**, then **Cell 6** to execute.
5. To do a real upload, set `DRY_RUN = False` in Cell 3 and re-run Cell 6.

## Supported Formats

GeoJSON, TopoJSON, Shapefile (ZIP/SHP), GeoPackage, KML/KMZ, FlatGeobuf, GeoParquet, GPX.

## Key Limits

- Required token scopes: `tilesets:read`, `tilesets:write`, `tilesets:list`.
- Default upload cap: 1 GB (opt into 20 GB with `USE_MAPBOX_FULL_UPLOAD_CAP = True`).
- Default zoom range: `4–8` (Mapbox supports `0–22`).

## Reading Output (Cell 6)

| Field | Meaning |
|---|---|
| `success` | Whether the workflow completed without error |
| `dry_run` | Simulation or real upload |
| `tileset_id` | Target tileset identifier |
| `steps` | Pipeline steps executed |
| `warnings` | Non-fatal issues to review |

## Colab Users

- Add `MAPBOX_ACCESS_TOKEN` and `MAPBOX_USERNAME` via the **Secrets** panel (key icon). Cell 3 auto-detects them.
- Upload files: `from google.colab import files; uploaded = files.upload()`, then set `SOURCE_FILE = Path('/content/<filename>')` in Cell 3.

## Cell 1 — Install MTU (Optional)

Install the `mtu` package into the current kernel. Skip this cell if `mtu` is already installed. Set `INSTALL_MTU = True` to enable.

In [ ]:
import subprocess
import sys

INSTALL_MTU = True
INSTALL_TARGET = "mtu"  # or pin a version, e.g. "mtu==0.1.0"

if INSTALL_MTU:
    print(f"Installing {INSTALL_TARGET} into: {sys.executable}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", INSTALL_TARGET])
    print("Done. Run Cell 2 to verify.")
else:
    print("Skipped. Set INSTALL_MTU = True to install.")

## Cell 2 — Check Dependencies

Verify that `mtu` is importable in the active kernel. If it's missing, go back and run Cell 1.

In [ ]:
import importlib.util

if importlib.util.find_spec('mtu'):
    print('✓ mtu is available.')
else:
    print('✗ mtu is NOT installed. Run Cell 1 with INSTALL_MTU = True, then re-run this cell.')

## Cell 3 — Settings

**This is the only cell you need to edit.** Set your file path, credentials, tileset info, and run options below.

Key inputs:
- `SOURCE_FILE` — path to your geospatial file.
- `MAPBOX_ACCESS_TOKEN` / `MAPBOX_USERNAME` — your Mapbox credentials (auto-detected from env or Colab Secrets).
- `DRY_RUN` — set to `False` for a real upload.

In [ ]:
import os
from pathlib import Path
from typing import Literal, TypedDict, cast

# ── USER INPUTS (edit these) ─────────────────────────────────

# Source file or URL.
SOURCE_FILE = Path(r'...\data.geojson')         # local file path
SOURCE_URL  = 'https://example.com/data.geojson' # remote URL
SOURCE_MODE_RAW = 'file'                          # 'file' or 'url'
FORMAT_HINT: str | None = None                    # None = auto-detect

# Mapbox credentials (auto-filled from env / Colab Secrets if blank).
MAPBOX_ACCESS_TOKEN = os.environ.get('MAPBOX_ACCESS_TOKEN', '')
MAPBOX_USERNAME     = os.environ.get('MAPBOX_USERNAME', '')

# Tileset info.
TILESET_ID   = 'demo-tileset-id'
TILESET_NAME = 'Demo Tileset'
LAYER_NAME   = 'data'
MIN_ZOOM = 4
MAX_ZOOM = 8

# Run mode.
DRY_RUN = True  # False = real upload

# Advanced (usually no changes needed).
USE_MAPBOX_FULL_UPLOAD_CAP = False   # True = 20 GB cap instead of 1 GB
CAPACITY_LIMIT_MB = 0.0
CAPACITY_USED_MB  = 0.0

# ── DO NOT EDIT BELOW THIS LINE ──────────────────────────────

APP_DEFAULT_UPLOAD_CAP_GB = 1.0
MAPBOX_FULL_UPLOAD_CAP_GB = 20.0
MAPBOX_ZOOM_MIN = 0
MAPBOX_ZOOM_MAX = 22


class Settings(TypedDict):
    mapbox_access_token: str
    mapbox_username: str
    dry_run: bool
    source_mode: Literal['file', 'url']
    source_file: Path
    source_url: str
    format_hint: str | None
    tileset_id: str
    tileset_name: str
    layer_name: str
    min_zoom: int
    max_zoom: int
    mapbox_zoom_min: int
    mapbox_zoom_max: int
    app_default_upload_cap_gb: float
    mapbox_full_upload_cap_gb: float
    use_mapbox_full_upload_cap: bool
    capacity_limit_mb: float
    capacity_used_mb: float


# Auto-detect Colab secrets.
_token_colab = _user_colab = ''
try:
    from google.colab import userdata  # type: ignore
    _token_colab = str(userdata.get('MAPBOX_ACCESS_TOKEN') or '')
    _user_colab = str(userdata.get('MAPBOX_USERNAME') or '')
except Exception:
    pass

if SOURCE_MODE_RAW not in {'file', 'url'}:
    raise ValueError("SOURCE_MODE_RAW must be 'file' or 'url'")
SOURCE_MODE = cast(Literal['file', 'url'], SOURCE_MODE_RAW)

SETTINGS: Settings = {
    'mapbox_access_token': _token_colab or MAPBOX_ACCESS_TOKEN,
    'mapbox_username': _user_colab or MAPBOX_USERNAME,
    'dry_run': DRY_RUN,
    'source_mode': SOURCE_MODE,
    'source_file': SOURCE_FILE,
    'source_url': SOURCE_URL,
    'format_hint': FORMAT_HINT,
    'tileset_id': TILESET_ID,
    'tileset_name': TILESET_NAME,
    'layer_name': LAYER_NAME,
    'min_zoom': MIN_ZOOM,
    'max_zoom': MAX_ZOOM,
    'mapbox_zoom_min': MAPBOX_ZOOM_MIN,
    'mapbox_zoom_max': MAPBOX_ZOOM_MAX,
    'app_default_upload_cap_gb': APP_DEFAULT_UPLOAD_CAP_GB,
    'mapbox_full_upload_cap_gb': MAPBOX_FULL_UPLOAD_CAP_GB,
    'use_mapbox_full_upload_cap': USE_MAPBOX_FULL_UPLOAD_CAP,
    'capacity_limit_mb': CAPACITY_LIMIT_MB,
    'capacity_used_mb': CAPACITY_USED_MB,
}

print(f'✓ Config loaded  dry_run={DRY_RUN}  mode={SOURCE_MODE}  zoom={MIN_ZOOM}-{MAX_ZOOM}')
print(f'  source: {SOURCE_FILE if SOURCE_MODE == "file" else SOURCE_URL}')
print(f'  token set: {bool(SETTINGS["mapbox_access_token"])}  username set: {bool(SETTINGS["mapbox_username"])}')

## Cell 4 — Import Libraries

Load required modules. No edits needed.

In [ ]:
import json
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

from mtu.uploader import TilesetConfig, TilesetUploader

## Cell 5 — Helper Functions

Reusable helpers for building config objects, the uploader, and previewing source data. No edits needed.

In [ ]:
def build_tileset_config(settings: Settings) -> TilesetConfig:
    return TilesetConfig(
        tileset_id=settings['tileset_id'],
        tileset_name=settings['tileset_name'],
        layer_name=settings['layer_name'],
        min_zoom=settings['min_zoom'],
        max_zoom=settings['max_zoom'],
    )


def build_uploader(settings: Settings) -> TilesetUploader:
    return TilesetUploader(
        access_token=settings['mapbox_access_token'],
        username=settings['mapbox_username'],
        validate_geometry=True,
        use_mapbox_full_upload_cap=settings['use_mapbox_full_upload_cap'],
    )


def get_active_upload_cap_gb(settings: Settings) -> float:
    return (
        settings['mapbox_full_upload_cap_gb']
        if settings['use_mapbox_full_upload_cap']
        else settings['app_default_upload_cap_gb']
    )


def describe_source(settings: Settings) -> None:
    cap_gb = get_active_upload_cap_gb(settings)
    cap_mb = cap_gb * 1024
    print(f'Active upload cap: {cap_gb} GB')

    if settings['source_mode'] == 'url':
        source_url = settings['source_url']
        print(f'URL source selected: {source_url}')
        parsed = urlparse(source_url)
        if not parsed.scheme or not parsed.netloc:
            raise ValueError('SOURCE_URL must be a valid absolute URL when source_mode is url')
        if settings['capacity_limit_mb'] > 0:
            print('Note: local size/capacity projection is skipped for URL sources.')
        return

    source_file = settings['source_file']
    if not source_file.exists():
        print(f'WARNING: file not found: {source_file}')
        return

    size_mb = source_file.stat().st_size / (1024 * 1024)
    print(f'Input size: {size_mb:.2f} MB')
    if size_mb > cap_mb:
        print('WARNING: input file exceeds active upload cap.')

    capacity_limit_mb = settings['capacity_limit_mb']
    if capacity_limit_mb > 0:
        projected = settings['capacity_used_mb'] + size_mb
        print(f'Projected usage: {projected:.2f} / {capacity_limit_mb:.2f} MB')
        if projected > capacity_limit_mb:
            print('WARNING: projected usage exceeds configured capacity limit.')

    if source_file.suffix.lower() not in {'.geojson', '.json'}:
        return

    try:
        import folium
        from IPython.display import display

        with source_file.open('r', encoding='utf-8') as f:
            geojson_data: Any = json.load(f)

        m = folium.Map(location=[0, 0], zoom_start=settings['min_zoom'], control_scale=True)
        layer = folium.GeoJson(geojson_data, name='source data')
        layer.add_to(m)

        try:
            bounds = layer.get_bounds()
            if bounds and len(bounds) == 2:
                sw, ne = bounds
                if all(v is not None for v in [sw[0], sw[1], ne[0], ne[1]]):
                    m.fit_bounds([[float(sw[0]), float(sw[1])], [float(ne[0]), float(ne[1])]])
        except Exception:
            pass

        display(m)
    except Exception as ex:
        print('Map preview skipped (folium not installed or invalid GeoJSON):', ex)

Active upload cap: 1 GB
Input size: 7.38 MB


## Cell 6 — Run Upload

Execute the upload (or dry run). Reviews source, builds config, and runs the pipeline. Re-run this cell after changing `DRY_RUN` in Cell 3.

In [ ]:
describe_source(SETTINGS)

if not SETTINGS['mapbox_access_token'] or not SETTINGS['mapbox_username']:
    raise ValueError(
        'Missing Mapbox credentials. Set MAPBOX_ACCESS_TOKEN and MAPBOX_USERNAME '
        'in Cell 3, then re-run from Cell 3.'
    )

config = build_tileset_config(SETTINGS)
uploader = build_uploader(SETTINGS)

if SETTINGS['source_mode'] == 'url':
    result = uploader.upload_from_url(
        url=SETTINGS['source_url'],
        config=config,
        format_hint=SETTINGS['format_hint'],
        dry_run=SETTINGS['dry_run'],
    )
else:
    result = uploader.upload_from_file(
        file_path=SETTINGS['source_file'],
        config=config,
        format_hint=SETTINGS['format_hint'],
        dry_run=SETTINGS['dry_run'],
    )

print(f'success:    {result.success}')
print(f'dry_run:    {result.dry_run}')
print(f'tileset_id: {result.tileset_id}')
print(f'steps:      {result.steps}')
print(f'warnings:   {result.warnings}')

success: True
dry_run: True
tileset_id: ocha-rosea-1.demo-tileset-id
steps: {'convert': True, 'validate': True}
warnings: []
